In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams
from sys import prefix
from utils import MusicDBPermDir
from pandas import Series, DataFrame
mp = MasterParams(verbose=True)
io = FileIO()
mdbpd = MusicDBPermDir()

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb


# DB-Specific

In [3]:
from lib import lastfm
mio   = lastfm.MusicDBIO(verbose=False)
apiio = lastfm.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath("LastFM")
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant LastFM DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/LastFM


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtistData   = lastfm.MusicDBIO(verbose=False).data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
localArtistInfo    = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistInfo".format(db.lower()))
oldToNew           = MusicDBData(path=permDir, fname="{0}oldToNewMap".format(db.lower()))
localAlbums        = MusicDBData(path=permDir, fname="{0}SearchedForLocalAlbums".format(db.lower()))
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Artist Search Data:        {0}".format(len(searchArtistData)))
#print("   Local Artist Search:       {0}".format(len(localArtists.get())))
print("   Local Artist Info:         {0}".format(len(localArtistInfo.get())))
print("   Old => New Map:            {0}".format(len(oldToNew.get())))
print("   Local Album Search:        {0}".format(len(localAlbums.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

LastFM Search Results
   Global Master Search:      140121
   Global Master Search Data: 0
   Artist Search Data:        2197579
   Local Artist Info:         141324
   Old => New Map:            65023
   Local Album Search:        144253
   Errors:                    0
   Known Summary IDs:         191104


# Move Files Around

In [ ]:
aid = lastfm.MusicDBID()

In [ ]:
aid = lastfm.MusicDBID()
#from tqdm.notebook.*
from tqdm._tqdm_notebook import tqdm_notebook
tqdm_notebook.pandas()
searchArtistData   = lastfm.MusicDBIO(verbose=False).data.getSearchArtistData()
searchArtistData["NAME"] = searchArtistData['name'].apply(aid.getArtistIDName)
#searchArtistData["ArtistPseudoIDOld"] = searchArtistData['name'].progress_apply(aid.getArtistPseudoIDOld)  #searchArtistData["ArtistID"] = searchArtistData['url'].apply(aid.getArtistID)
#searchArtistData["ArtistPseudoID"] = searchArtistData['name'].progress_apply(aid.getArtistPseudoID)  #searchArtistData["ArtistID"] = searchArtistData['url'].apply(aid.getArtistID)
#possibleArtistsData = searchArtistData['name']
#possibleArtistsData.index = searchArtistData['ArtistPseudoID']

## Artist

In [ ]:
aid = lastfm.MusicDBID()
#localArtistsInfoDict = localArtistInfo.get()
#oldToNewMap = {}
print("Local Artist Info:         {0}".format(len(localArtistsInfoDict)))
print("="*175)
for modVal in range(100):
    files = DirInfo("/Volumes/Piggy/Discog/artists-lastfmold/{0}".format(modVal)).glob("*.p")
    for i,ifile in enumerate(files):
        oldID = ifile.stem
        name = io.get(ifile).get('name')
        psid = aid.getArtistPseudoID(name)
        if psid is None:
            continue
        oldToNewMap[oldID] = psid
        artistInfoFilenameOld = FileInfo(ifile)
        artistInfoFilenameNew = mio.data.getRawArtistInfoFilename(mio.mv.get(psid), psid)
        localArtistsInfoDict[psid] = name
        artistInfoFilenameOld.mvFile(artistInfoFilenameNew)
        if i % 100 == 0:
            print("="*150)
            print("Saving {0} Known Artist ID Lookups".format(len(localArtistsInfoDict)))
            localArtistInfo.save(data=localArtistsInfoDict)
            print("Saving {0} Old=>New Artist ID Lookups".format(len(oldToNewMap)))
            io.save(idata=oldToNewMap, ifile="oldToNewMap.p")            
            print("="*150)

    print("="*150)
    print("Saving {0} Known Artist ID Lookups".format(len(localArtistsInfoDict)))
    localArtistInfo.save(data=localArtistsInfoDict)
    print("Saving {0} Old=>New Artist ID Lookups".format(len(oldToNewMap)))
    io.save(idata=oldToNewMap, ifile="oldToNewMap.p")                
    print("="*150)

In [ ]:
artistInfoData = io.get(ifile)

In [ ]:
localArtistsInfoDict = localArtistInfo.get()
missing = io.get('missing.p')
print("Local Artist Info:         {0}".format(len(localArtistsInfoDict)))
print("Missing Artist Map:        {0}".format(len(missing)))
print("="*175)
nMiss = 0
for i,(idx,row) in enumerate(searchArtistData.iterrows()):
    psidOld = aid.getArtistPseudoIDOld(row["name"])
    psid    = aid.getArtistPseudoID(row["name"])
    if localArtistsInfoDict.get(psid) is not None:
        continue
    if missing.get(psid) is not None:
        continue
    name    = row["name"]
    artistInfoFilenameOld = mio.data.getRawArtistInfoFilename(mio.mv.get(psidOld), psidOld)
    artistInfoFilenameOld = FileInfo(artistInfoFilenameOld.str.replace("lastfm", "lastfmold"))
    if artistInfoFilenameOld.exists():
        artistInfoFilenameNew = mio.data.getRawArtistInfoFilename(mio.mv.get(psid), psid)
        #print("{0: <50}{1: >20}{2: >70}{3: >10}".format(name,psidOld,artistInfoFilenameOld.str,artistInfoFilenameOld.exists()))
        #print("{0: <50}{1: >20}{2: >70}{3: >10}".format(row["NAME"],psid,artistInfoFilenameNew.str,""))
        #print("{0: <50}{1: <20}".format(name,psid),end="")
        localArtistsInfoDict[psid] = name
        artistInfoFilenameOld.mvFile(artistInfoFilenameNew)
        nMiss = 0
    else:
        missing[psid] = name
        print("{0: <50}{1: <20} Missing".format(name,psid))
        nMiss += 1
        
    if nMiss > 1000:
        break
        
    if i % 2500 == 0:
        print("="*150)
        print("Saving {0} Known Artist ID Lookups".format(len(localArtistsInfoDict)))
        localArtistInfo.save(data=localArtistsInfoDict)
        print("Saving {0} Missing Artist ID Lookups".format(len(missing)))
        io.save(idata=missing, ifile="missing.p")
        print("="*150)
        
        
print("Saving {0} Known Artist ID Lookups".format(len(localArtistsInfoDict)))
localArtistInfo.save(data=localArtistsInfoDict)
print("Saving {0} Missing Artist ID Lookups".format(len(missing)))
io.save(idata=missing, ifile="missing.p")

## Albums

In [13]:
aid = lastfm.MusicDBID()
localAlbumsDict = localAlbums.get()
localArtistsInfoDict = localArtistInfo.get()
oldToNewMap = oldToNew.get()
print("Local Albums:         {0}".format(len(localAlbumsDict)))
print("Old => New Map:       {0}".format(len(oldToNewMap)))
print("="*175)
newData = {}
for modVal in range(2,100):
    newData[modVal] = 0
    files = DirInfo("/Volumes/Piggy/Discog/artists-lastfmold/{0}/albums".format(modVal)).glob("*.p")
    print("="*150)

    for i,ifile in enumerate(files):
        oldID = ifile.stem
        psid = oldToNewMap.get(oldID)
        if psid is None:
            continue
        name = localArtistsInfoDict.get(psid)
        if name is None:
            continue
        albumsFilenameOld = FileInfo(ifile)
        albumsFilenameNew = mio.data.getRawArtistAlbumFilename(mio.mv.get(psid), psid)
        localAlbumsDict[psid] = name
        albumsFilenameOld.mvFile(albumsFilenameNew, debug=False)
        newData[modVal] += 1
        if (i+1) % 100 == 0:
            print("  ===> Saving {0} [{1}/{2}] Known Artist ID Lookups".format(len(localAlbumsDict), newData[modVal], sum(newData.values())))
            localAlbums.save(data=localAlbumsDict)
    print("="*150)
    print("Saving {0} [{1}/{2}] Known Artist ID Lookups".format(len(localAlbumsDict), newData[modVal], sum(newData.values())))
    localAlbums.save(data=localAlbumsDict)
    print("="*150)
    print("")

Local Albums:         105385
Old => New Map:       65023
Running glob(/Volumes/Piggy/Discog/artists-lastfmold/2/albums/*.p)
  ===> Saving 105416 [31/31] Known Artist ID Lookups
  ===> Saving 105452 [67/67] Known Artist ID Lookups
  ===> Saving 105492 [108/108] Known Artist ID Lookups
  ===> Saving 105529 [146/146] Known Artist ID Lookups
  ===> Saving 105591 [209/209] Known Artist ID Lookups
Saving 105774 [398/398] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/3/albums/*.p)
  ===> Saving 105873 [100/498] Known Artist ID Lookups
Saving 106165 [396/794] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/4/albums/*.p)
  ===> Saving 106227 [62/856] Known Artist ID Lookups
  ===> Saving 106275 [110/904] Known Artist ID Lookups
  ===> Saving 106473 [309/1103] Known Artist ID Lookups
Saving 106597 [434/1228] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/5/albums/*.p)
  ===> Saving 106628 [31/1259] Known

  ===> Saving 109430 [174/4092] Known Artist ID Lookups
  ===> Saving 109469 [213/4131] Known Artist ID Lookups
  ===> Saving 109661 [408/4326] Known Artist ID Lookups
Saving 109668 [415/4333] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/13/albums/*.p)
  ===> Saving 109703 [37/4370] Known Artist ID Lookups
  ===> Saving 109768 [105/4438] Known Artist ID Lookups
  ===> Saving 109802 [140/4473] Known Artist ID Lookups
  ===> Saving 109859 [198/4531] Known Artist ID Lookups
  ===> Saving 109923 [262/4595] Known Artist ID Lookups
Saving 110052 [391/4724] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/14/albums/*.p)
  ===> Saving 110166 [118/4842] Known Artist ID Lookups
  ===> Saving 110263 [215/4939] Known Artist ID Lookups
Saving 110461 [415/5139] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/15/albums/*.p)
  ===> Saving 110489 [28/5167] Known Artist ID Lookups
  ===> Saving 110558 [98/5237] K

  ===> Saving 113884 [256/8600] Known Artist ID Lookups
Saving 114040 [415/8759] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/24/albums/*.p)
  ===> Saving 114144 [104/8863] Known Artist ID Lookups
  ===> Saving 114237 [200/8959] Known Artist ID Lookups
  ===> Saving 114342 [306/9065] Known Artist ID Lookups
Saving 114406 [372/9131] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/25/albums/*.p)
  ===> Saving 114442 [37/9168] Known Artist ID Lookups
  ===> Saving 114512 [107/9238] Known Artist ID Lookups
  ===> Saving 114662 [258/9389] Known Artist ID Lookups
  ===> Saving 114697 [294/9425] Known Artist ID Lookups
Saving 114784 [381/9512] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/26/albums/*.p)
  ===> Saving 114951 [168/9680] Known Artist ID Lookups
Saving 115164 [383/9895] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/27/albums/*.p)
  ===> Saving 115200 [36/

  ===> Saving 118233 [330/13006] Known Artist ID Lookups
Saving 118295 [394/13070] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/35/albums/*.p)
  ===> Saving 118411 [118/13188] Known Artist ID Lookups
  ===> Saving 118668 [376/13446] Known Artist ID Lookups
Saving 118694 [402/13472] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/36/albums/*.p)
  ===> Saving 118732 [40/13512] Known Artist ID Lookups
  ===> Saving 118972 [283/13755] Known Artist ID Lookups
  ===> Saving 119023 [335/13807] Known Artist ID Lookups
  ===> Saving 119055 [367/13839] Known Artist ID Lookups
  ===> Saving 119088 [403/13875] Known Artist ID Lookups
Saving 119094 [409/13881] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/37/albums/*.p)
  ===> Saving 119174 [82/13963] Known Artist ID Lookups
  ===> Saving 119220 [129/14010] Known Artist ID Lookups
  ===> Saving 119284 [193/14074] Known Artist ID Lookups
  ===> Saving 1194


Running glob(/Volumes/Piggy/Discog/artists-lastfmold/45/albums/*.p)
  ===> Saving 122372 [36/17191] Known Artist ID Lookups
  ===> Saving 122526 [192/17347] Known Artist ID Lookups
  ===> Saving 122563 [229/17384] Known Artist ID Lookups
  ===> Saving 122590 [257/17412] Known Artist ID Lookups
  ===> Saving 122622 [290/17445] Known Artist ID Lookups
  ===> Saving 122698 [367/17522] Known Artist ID Lookups
Saving 122737 [407/17562] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/46/albums/*.p)
  ===> Saving 123023 [288/17850] Known Artist ID Lookups
  ===> Saving 123088 [355/17917] Known Artist ID Lookups
Saving 123166 [434/17996] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/47/albums/*.p)
  ===> Saving 123199 [33/18029] Known Artist ID Lookups
  ===> Saving 123409 [246/18242] Known Artist ID Lookups
  ===> Saving 123443 [281/18277] Known Artist ID Lookups
  ===> Saving 123541 [382/18378] Known Artist ID Lookups
Saving 1235

  ===> Saving 126462 [101/21338] Known Artist ID Lookups
  ===> Saving 126562 [203/21440] Known Artist ID Lookups
  ===> Saving 126692 [333/21570] Known Artist ID Lookups
  ===> Saving 126757 [400/21637] Known Artist ID Lookups
Saving 126771 [414/21651] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/56/albums/*.p)
  ===> Saving 126833 [65/21716] Known Artist ID Lookups
  ===> Saving 126876 [108/21759] Known Artist ID Lookups
  ===> Saving 126937 [170/21821] Known Artist ID Lookups
  ===> Saving 127000 [233/21884] Known Artist ID Lookups
  ===> Saving 127032 [265/21916] Known Artist ID Lookups
  ===> Saving 127066 [300/21951] Known Artist ID Lookups
  ===> Saving 127176 [411/22062] Known Artist ID Lookups
Saving 127187 [422/22073] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/57/albums/*.p)
  ===> Saving 127296 [112/22185] Known Artist ID Lookups
  ===> Saving 127328 [144/22217] Known Artist ID Lookups
  ===> Saving 127393 [


Running glob(/Volumes/Piggy/Discog/artists-lastfmold/66/albums/*.p)
  ===> Saving 130811 [105/25759] Known Artist ID Lookups
  ===> Saving 130942 [238/25892] Known Artist ID Lookups
  ===> Saving 130968 [265/25919] Known Artist ID Lookups
  ===> Saving 131086 [384/26038] Known Artist ID Lookups
Saving 131120 [418/26072] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/67/albums/*.p)
  ===> Saving 131302 [185/26257] Known Artist ID Lookups
  ===> Saving 131403 [286/26358] Known Artist ID Lookups
  ===> Saving 131550 [435/26507] Known Artist ID Lookups
Saving 131550 [435/26507] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/68/albums/*.p)
  ===> Saving 131690 [140/26647] Known Artist ID Lookups
  ===> Saving 131805 [255/26762] Known Artist ID Lookups
Saving 131904 [355/26862] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/69/albums/*.p)
  ===> Saving 132005 [103/26965] Known Artist ID Lookups
  ==

  ===> Saving 135070 [296/30071] Known Artist ID Lookups
Saving 135157 [383/30158] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/77/albums/*.p)
  ===> Saving 135185 [28/30186] Known Artist ID Lookups
  ===> Saving 135256 [100/30258] Known Artist ID Lookups
  ===> Saving 135286 [131/30289] Known Artist ID Lookups
  ===> Saving 135398 [245/30403] Known Artist ID Lookups
Saving 135549 [397/30555] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/78/albums/*.p)
  ===> Saving 135607 [60/30615] Known Artist ID Lookups
  ===> Saving 135863 [321/30876] Known Artist ID Lookups
Saving 135916 [374/30929] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/79/albums/*.p)
  ===> Saving 135981 [65/30994] Known Artist ID Lookups
  ===> Saving 136024 [108/31037] Known Artist ID Lookups
  ===> Saving 136178 [267/31196] Known Artist ID Lookups
  ===> Saving 136250 [340/31269] Known Artist ID Lookups
Saving 136313 [405/

  ===> Saving 139140 [30/34205] Known Artist ID Lookups
  ===> Saving 139434 [325/34500] Known Artist ID Lookups
Saving 139487 [378/34553] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/88/albums/*.p)
  ===> Saving 139606 [119/34672] Known Artist ID Lookups
  ===> Saving 139645 [158/34711] Known Artist ID Lookups
  ===> Saving 139711 [224/34777] Known Artist ID Lookups
  ===> Saving 139836 [350/34903] Known Artist ID Lookups
  ===> Saving 139861 [376/34929] Known Artist ID Lookups
Saving 139899 [414/34967] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/89/albums/*.p)
  ===> Saving 140004 [108/35075] Known Artist ID Lookups
  ===> Saving 140176 [286/35253] Known Artist ID Lookups
Saving 140292 [403/35370] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/90/albums/*.p)
  ===> Saving 140407 [119/35489] Known Artist ID Lookups
  ===> Saving 140515 [228/35598] Known Artist ID Lookups
  ===> Saving 140

  ===> Saving 143527 [31/38669] Known Artist ID Lookups
  ===> Saving 143700 [205/38843] Known Artist ID Lookups
  ===> Saving 143768 [273/38911] Known Artist ID Lookups
  ===> Saving 143795 [300/38938] Known Artist ID Lookups
  ===> Saving 143831 [336/38974] Known Artist ID Lookups
  ===> Saving 143862 [367/39005] Known Artist ID Lookups
Saving 143911 [417/39055] Known Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/99/albums/*.p)
  ===> Saving 144071 [165/39220] Known Artist ID Lookups
  ===> Saving 144102 [196/39251] Known Artist ID Lookups
  ===> Saving 144128 [224/39279] Known Artist ID Lookups
  ===> Saving 144220 [320/39375] Known Artist ID Lookups
Saving 144253 [354/39409] Known Artist ID Lookups



In [21]:
aid = lastfm.MusicDBID()
localAlbumsDict = localAlbums.get()
localArtistsInfoDict = localArtistInfo.get()
oldToNewMap = oldToNew.get()
print("Local Albums:         {0}".format(len(localAlbumsDict)))
print("Old => New Map:       {0}".format(len(oldToNewMap)))
print("="*175)
newData = {}
for modVal in range(100):
    newData[modVal] = 0
    files = DirInfo("/Volumes/Piggy/Discog/artists-lastfmold/{0}/albums".format(modVal)).glob("*.p")
    print("="*150)

    for i,ifile in enumerate(files):
        oldID = ifile.stem
        psid = oldToNewMap.get(oldID)
        name = localArtistsInfoDict.get(psid) if psid is not None else None
        if name is None:
            data = io.get(ifile)
            cntr = Counter()
            for k,v in data.items():
                for rec in v:
                    cntr[rec["artistName"]] += 1
            if len(cntr) > 0:
                name = cntr.most_common()[0][0]
                psid = aid.getArtistPseudoID(name)
                oldToNewMap[oldID] = psid
            else:
                continue
        albumsFilenameOld = FileInfo(ifile)
        albumsFilenameNew = mio.data.getRawArtistAlbumFilename(mio.mv.get(psid), psid)
        localAlbumsDict[psid] = name
        albumsFilenameOld.mvFile(albumsFilenameNew, debug=False)
        newData[modVal] += 1
        if (i+1) % 100 == 0:
            print("  ===> Saving {0} [{1}/{2}] Known Artist ID Lookups".format(len(localAlbumsDict), newData[modVal], sum(newData.values())))
            localAlbums.save(data=localAlbumsDict)
            print("  ===> Saving {0} Old=>New Artist ID Lookups".format(len(oldToNewMap)))
            io.save(idata=oldToNewMap, ifile="oldToNewMap.p")                
    print("="*150)
    print("Saving {0} [{1}/{2}] Known Artist ID Lookups".format(len(localAlbumsDict), newData[modVal], sum(newData.values())))
    localAlbums.save(data=localAlbumsDict)
    print("Saving {0} Old=>New Artist ID Lookups".format(len(oldToNewMap)))
    io.save(idata=oldToNewMap, ifile="oldToNewMap.p")                
    print("="*150)
    print("")

Local Albums:         144684
Old => New Map:       65023
Running glob(/Volumes/Piggy/Discog/artists-lastfmold/0/albums/*.p)
Saving 144684 [0/0] Known Artist ID Lookups
Saving 65023 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/1/albums/*.p)
  ===> Saving 144747 [63/63] Known Artist ID Lookups
Saving 65086 Old=>New Artist ID Lookups
  ===> Saving 144798 [116/116] Known Artist ID Lookups
Saving 65139 Old=>New Artist ID Lookups
  ===> Saving 144906 [225/225] Known Artist ID Lookups
Saving 65247 Old=>New Artist ID Lookups
  ===> Saving 144957 [276/276] Known Artist ID Lookups
Saving 65298 Old=>New Artist ID Lookups
  ===> Saving 145093 [412/412] Known Artist ID Lookups
Saving 65434 Old=>New Artist ID Lookups
Saving 145135 [454/454] Known Artist ID Lookups
Saving 65475 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/2/albums/*.p)
  ===> Saving 145362 [228/682] Known Artist ID Lookups
Saving 65703 Old=>New Artist ID Lookups


Saving 68399 Old=>New Artist ID Lookups
  ===> Saving 148112 [178/3459] Known Artist ID Lookups
Saving 68466 Old=>New Artist ID Lookups
  ===> Saving 148153 [219/3500] Known Artist ID Lookups
Saving 68507 Old=>New Artist ID Lookups
  ===> Saving 148200 [266/3547] Known Artist ID Lookups
Saving 68554 Old=>New Artist ID Lookups
  ===> Saving 148258 [324/3605] Known Artist ID Lookups
Saving 68612 Old=>New Artist ID Lookups
  ===> Saving 148383 [451/3732] Known Artist ID Lookups
Saving 68739 Old=>New Artist ID Lookups
Saving 148388 [456/3737] Known Artist ID Lookups
Saving 68744 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/9/albums/*.p)
  ===> Saving 148510 [122/3859] Known Artist ID Lookups
Saving 68865 Old=>New Artist ID Lookups
  ===> Saving 148605 [218/3955] Known Artist ID Lookups
Saving 68961 Old=>New Artist ID Lookups
  ===> Saving 148663 [277/4014] Known Artist ID Lookups
Saving 69020 Old=>New Artist ID Lookups
  ===> Saving 148725 [340/4077] Kno

Saving 72222 Old=>New Artist ID Lookups
  ===> Saving 151922 [412/7303] Known Artist ID Lookups
Saving 72287 Old=>New Artist ID Lookups
Saving 151959 [449/7340] Known Artist ID Lookups
Saving 72324 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/17/albums/*.p)
  ===> Saving 152014 [56/7396] Known Artist ID Lookups
Saving 72380 Old=>New Artist ID Lookups
  ===> Saving 152066 [108/7448] Known Artist ID Lookups
Saving 72432 Old=>New Artist ID Lookups
  ===> Saving 152121 [163/7503] Known Artist ID Lookups
Saving 72487 Old=>New Artist ID Lookups
  ===> Saving 152174 [216/7556] Known Artist ID Lookups
Saving 72540 Old=>New Artist ID Lookups
  ===> Saving 152283 [327/7667] Known Artist ID Lookups
Saving 72651 Old=>New Artist ID Lookups
  ===> Saving 152339 [384/7724] Known Artist ID Lookups
Saving 72708 Old=>New Artist ID Lookups
Saving 152396 [442/7782] Known Artist ID Lookups
Saving 72766 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artist

  ===> Saving 155329 [107/10745] Known Artist ID Lookups
Saving 75716 Old=>New Artist ID Lookups
  ===> Saving 155489 [267/10905] Known Artist ID Lookups
Saving 75872 Old=>New Artist ID Lookups
  ===> Saving 155549 [329/10967] Known Artist ID Lookups
Saving 75934 Old=>New Artist ID Lookups
Saving 155693 [476/11114] Known Artist ID Lookups
Saving 76081 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/25/albums/*.p)
  ===> Saving 155754 [62/11176] Known Artist ID Lookups
Saving 76143 Old=>New Artist ID Lookups
  ===> Saving 155804 [112/11226] Known Artist ID Lookups
Saving 76193 Old=>New Artist ID Lookups
  ===> Saving 155905 [214/11328] Known Artist ID Lookups
Saving 76295 Old=>New Artist ID Lookups
  ===> Saving 155957 [266/11380] Known Artist ID Lookups
Saving 76346 Old=>New Artist ID Lookups
  ===> Saving 156073 [383/11497] Known Artist ID Lookups
Saving 76463 Old=>New Artist ID Lookups
Saving 156132 [443/11557] Known Artist ID Lookups
Saving 76522 Old

Saving 79269 Old=>New Artist ID Lookups
  ===> Saving 158931 [122/14375] Known Artist ID Lookups
Saving 79329 Old=>New Artist ID Lookups
  ===> Saving 158984 [175/14428] Known Artist ID Lookups
Saving 79382 Old=>New Artist ID Lookups
  ===> Saving 159030 [222/14475] Known Artist ID Lookups
Saving 79428 Old=>New Artist ID Lookups
  ===> Saving 159087 [279/14532] Known Artist ID Lookups
Saving 79485 Old=>New Artist ID Lookups
Saving 159263 [459/14712] Known Artist ID Lookups
Saving 79663 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/33/albums/*.p)
  ===> Saving 159324 [61/14773] Known Artist ID Lookups
Saving 79723 Old=>New Artist ID Lookups
  ===> Saving 159372 [110/14822] Known Artist ID Lookups
Saving 79772 Old=>New Artist ID Lookups
  ===> Saving 159487 [225/14937] Known Artist ID Lookups
Saving 79887 Old=>New Artist ID Lookups
  ===> Saving 159542 [281/14993] Known Artist ID Lookups
Saving 79943 Old=>New Artist ID Lookups
  ===> Saving 159671 [411/

...

Saving 93848 Old=>New Artist ID Lookups
  ===> Saving 173396 [470/29021] Known Artist ID Lookups
Saving 93902 Old=>New Artist ID Lookups
Saving 173397 [471/29022] Known Artist ID Lookups
Saving 93903 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/63/albums/*.p)
  ===> Saving 173570 [174/29196] Known Artist ID Lookups
Saving 94076 Old=>New Artist ID Lookups
  ===> Saving 173676 [283/29305] Known Artist ID Lookups
Saving 94185 Old=>New Artist ID Lookups
  ===> Saving 173728 [336/29358] Known Artist ID Lookups
Saving 94238 Old=>New Artist ID Lookups
Saving 173820 [430/29452] Known Artist ID Lookups
Saving 94332 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/64/albums/*.p)
  ===> Saving 173876 [56/29508] Known Artist ID Lookups
Saving 94387 Old=>New Artist ID Lookups
  ===> Saving 173987 [168/29620] Known Artist ID Lookups
Saving 94499 Old=>New Artist ID Lookups
  ===> Saving 174083 [265/29717] Known Artist ID Lookups
Savi

  ===> Saving 177152 [64/32827] Known Artist ID Lookups
Saving 97688 Old=>New Artist ID Lookups
  ===> Saving 177207 [119/32882] Known Artist ID Lookups
Saving 97743 Old=>New Artist ID Lookups
  ===> Saving 177380 [296/33059] Known Artist ID Lookups
Saving 97920 Old=>New Artist ID Lookups
  ===> Saving 177508 [428/33191] Known Artist ID Lookups
Saving 98052 Old=>New Artist ID Lookups
Saving 177561 [483/33246] Known Artist ID Lookups
Saving 98107 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/72/albums/*.p)
  ===> Saving 177679 [118/33364] Known Artist ID Lookups
Saving 98225 Old=>New Artist ID Lookups
  ===> Saving 177727 [167/33413] Known Artist ID Lookups
Saving 98274 Old=>New Artist ID Lookups
  ===> Saving 177776 [217/33463] Known Artist ID Lookups
Saving 98324 Old=>New Artist ID Lookups
  ===> Saving 177892 [333/33579] Known Artist ID Lookups
Saving 98439 Old=>New Artist ID Lookups
Saving 178028 [470/33716] Known Artist ID Lookups
Saving 98576 Old


Running glob(/Volumes/Piggy/Discog/artists-lastfmold/79/albums/*.p)
  ===> Saving 180805 [57/36545] Known Artist ID Lookups
Saving 101399 Old=>New Artist ID Lookups
  ===> Saving 180861 [115/36603] Known Artist ID Lookups
Saving 101456 Old=>New Artist ID Lookups
  ===> Saving 180961 [216/36704] Known Artist ID Lookups
Saving 101556 Old=>New Artist ID Lookups
  ===> Saving 181074 [331/36819] Known Artist ID Lookups
Saving 101671 Old=>New Artist ID Lookups
  ===> Saving 181132 [391/36879] Known Artist ID Lookups
Saving 101731 Old=>New Artist ID Lookups
  ===> Saving 181194 [453/36941] Known Artist ID Lookups
Saving 101793 Old=>New Artist ID Lookups
Saving 181196 [455/36943] Known Artist ID Lookups
Saving 101795 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/80/albums/*.p)
  ===> Saving 181262 [67/37010] Known Artist ID Lookups
Saving 101862 Old=>New Artist ID Lookups
  ===> Saving 181429 [237/37180] Known Artist ID Lookups
Saving 102032 Old=>New Artist 

Saving 104761 Old=>New Artist ID Lookups
  ===> Saving 184230 [280/40026] Known Artist ID Lookups
Saving 104865 Old=>New Artist ID Lookups
  ===> Saving 184282 [335/40081] Known Artist ID Lookups
Saving 104920 Old=>New Artist ID Lookups
  ===> Saving 184341 [394/40140] Known Artist ID Lookups
Saving 104979 Old=>New Artist ID Lookups
  ===> Saving 184412 [467/40213] Known Artist ID Lookups
Saving 105052 Old=>New Artist ID Lookups
Saving 184429 [484/40230] Known Artist ID Lookups
Saving 105069 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/87/albums/*.p)
  ===> Saving 184642 [215/40445] Known Artist ID Lookups
Saving 105284 Old=>New Artist ID Lookups
  ===> Saving 184704 [279/40509] Known Artist ID Lookups
Saving 105348 Old=>New Artist ID Lookups
  ===> Saving 184832 [409/40639] Known Artist ID Lookups
Saving 105476 Old=>New Artist ID Lookups
Saving 184877 [455/40685] Known Artist ID Lookups
Saving 105522 Old=>New Artist ID Lookups

Running glob(/Volumes

Saving 108439 Old=>New Artist ID Lookups
  ===> Saving 187811 [171/43672] Known Artist ID Lookups
Saving 108488 Old=>New Artist ID Lookups
  ===> Saving 187860 [222/43723] Known Artist ID Lookups
Saving 108539 Old=>New Artist ID Lookups
  ===> Saving 187912 [276/43777] Known Artist ID Lookups
Saving 108593 Old=>New Artist ID Lookups
  ===> Saving 188026 [392/43893] Known Artist ID Lookups
Saving 108709 Old=>New Artist ID Lookups
Saving 188105 [471/43972] Known Artist ID Lookups
Saving 108788 Old=>New Artist ID Lookups

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/95/albums/*.p)
  ===> Saving 188165 [60/44032] Known Artist ID Lookups
Saving 108848 Old=>New Artist ID Lookups
  ===> Saving 188223 [120/44092] Known Artist ID Lookups
Saving 108906 Old=>New Artist ID Lookups
  ===> Saving 188274 [171/44143] Known Artist ID Lookups
Saving 108957 Old=>New Artist ID Lookups
  ===> Saving 188316 [216/44188] Known Artist ID Lookups
Saving 109002 Old=>New Artist ID Lookups
  ===> Saving 18

In [19]:
localAlbums.save(data=localAlbumsDict)

In [14]:
for k,v in albumData.items():
    print(v.most_common()[0])
    break

('5Bugs', 1418)


In [9]:
from collections import Counter
modVal = 0
files = list(DirInfo("/Volumes/Piggy/Discog/artists-lastfmold/{0}/albums".format(modVal)).glob("*.p"))
albumData = {}
for ifile in files:
    data = io.get(ifile)
    cntr = Counter()
    for k,v in data.items():
        for rec in v:
            cntr[rec["artistName"]] += 1
    albumData[ifile] = cntr

Running glob(/Volumes/Piggy/Discog/artists-lastfmold/0/albums/*.p)


In [ ]:
aid = lastfm.MusicDBID()
localAlbumsDict = localAlbums.get()
localArtistsInfoDict = localArtistInfo.get()
oldToNewMap = oldToNew.get()
print("Local Albums:         {0}".format(len(localAlbumsDict)))
print("Old => New Map:       {0}".format(len(oldToNewMap)))
print("="*175)
newData = {}
for modVal in range(2,100):
    newData[modVal] = 0
    files = DirInfo("/Volumes/Piggy/Discog/artists-lastfmold/{0}/albums".format(modVal)).glob("*.p")
    print("="*150)

In [ ]:
localAlbumsDict = localAlbums.get()
missingAlbums = io.get('missingAlbums.p')
print("Local Albums:         {0}".format(len(localAlbumsDict)))
print("Missing Album Map:    {0}".format(len(missingAlbums)))
print("="*175)
nMiss = 0

for i,(idx,row) in enumerate(searchArtistData.iterrows()):
    psidOld = aid.getArtistPseudoIDOld(row["name"])
    psid    = aid.getArtistPseudoID(row["name"])
    if localAlbumsDict.get(psid) is not None:
        continue
    if missingAlbums.get(psid) is not None:
        continue
    name    = row["name"]
    albumsFilenameOld = mio.data.getRawArtistAlbumFilename(mio.mv.get(psidOld), psidOld)
    albumsFilenameOld = FileInfo(albumsFilenameOld.str.replace("lastfm", "lastfmold"))
    if albumsFilenameOld.exists():
        albumsFilenameNew = mio.data.getRawArtistAlbumFilename(mio.mv.get(psid), psid)
        localAlbumsDict[psid] = name
        albumsFilenameOld.mvFile(albumsFilenameNew, debug=False)
        nMiss = 0
    else:
        missingAlbums[psid] = name
        print("{0: <50}{1: <20} Missing".format(name,psid))
        nMiss += 1
        
    if nMiss > 1000:
        break
        
    if i % 1000 == 0:
        print("="*150)
        print("Saving {0} Known Artist ID Lookups".format(len(localAlbumsDict)))
        localAlbums.save(data=localAlbumsDict)
        print("Saving {0} Missing Artist ID Lookups".format(len(missingAlbums)))
        io.save(idata=missingAlbums, ifile="missingAlbums.p")
        print("="*150)
        

print("Saving {0} Known Album Artist ID Lookups".format(len(localAlbumsDict)))
localAlbums.save(data=localAlbumsDict)
print("Saving {0} Missing Album Artist ID Lookups".format(len(missingAlbums)))
io.save(idata=missingAlbums, ifile="missingAlbums.p")

# Download Artist Search Data

In [ ]:
mio   = lastfm.MusicDBIO(verbose=False)
apiio = lastfm.RawAPIData(debug=True)

## Find Artists To Download

In [ ]:
from musicdb import MusicDBIO
mdbio = MusicDBIO()
mmeDF = mdbio.getData()
unmatchedArtistNames = Series(mmeDF["ArtistName"].unique())
searchedForMasterArtists = masterArtists.get()
artistNamesToGet = unmatchedArtistNames[~unmatchedArtistNames.isin(searchedForMasterArtists.keys())]

print("{0} Search Results".format(db))
print("   Known Master Artist Names:    {0}".format(len(unmatchedArtistNames)))
print("   Known Spotify Artist Names:   {0}".format(len(searchedForMasterArtists)))
print("   Artist Names To Get:          {0}".format(len(artistNamesToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "2:30pm")
tt = TermTime("today", "9:15pm")
n  = 0

searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if searchedForMasterArtists.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchResults(artistName=artistName)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((idx,artistName)))
        searchedForMasterArtists[artistName] = True
        apiio.sleep(3.5)
        continue
        
    searchedForMasterArtistsData[artistName] = response
    searchedForMasterArtists[artistName] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = DataFrame(getFlatList([v for k,v in mad.items()]))
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = lastfm.MusicDBIO(verbose=False).data.getSearchArtistData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['listeners'] = searchDF['listeners'].astype(int)
    searchDF = searchDF.sort_values(by='listeners', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    lastfm.MusicDBIO(verbose=False).data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

In [ ]:
masterArtistsData.save(data={})

# Download Artist Info Data

In [ ]:
aid = lastfm.MusicDBID()
from tqdm._tqdm_notebook import tqdm_notebook
tqdm_notebook.pandas()
searchArtistData["ArtistPseudoID"] = searchArtistData['name'].progress_apply(aid.getArtistPseudoID)  #searchArtistData["ArtistID"] = searchArtistData['url'].apply(aid.getArtistID)
possibleArtistsData = searchArtistData['name']
possibleArtistsData.index = searchArtistData['ArtistPseudoID']

In [ ]:
print("{0} Search Results".format(db))
print("   Possible Artist Info:   {0}".format(possibleArtistsData.shape[0]))
knownArtistInfo = localArtistInfo.get()
print("   Downloaded Artist Info: {0}".format(len(knownArtistInfo)))
artistInfoToGet = possibleArtistsData[~possibleArtistsData.index.isin(knownArtistInfo.keys())]
print("   Artist Info To Get:     {0}".format(len(artistInfoToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "5:30pm")
#tt = TermTime("today", "7:15pm")
tt = TermTime("today", "9:15pm")
n  = 0

searchedForLocalArtistInfo = localArtistInfo.get()
searchedForErrors          = errors.get()
for i,(artistPseudoID,artistName) in enumerate(artistInfoToGet.iteritems()):
    if searchedForLocalArtistInfo.get(artistPseudoID) is not None:
        continue
    if searchedForErrors.get(artistPseudoID) is not None:
        continue

    modVal=mio.mv.get(artistPseudoID)
    if mio.data.getRawArtistInfoFilename(modVal, artistPseudoID).exists():
        searchedForLocalArtistInfo[artistPseudoID] = artistName
        continue

    response = apiio.getArtistInfo(artistName=artistName)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((idx,artistName)))
        searchedForErrors[artistPseudoID] = artistName
        apiio.sleep(3.5)
        continue
        
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=artistPseudoID)        
    searchedForLocalArtistInfo[artistPseudoID] = artistName
    apiio.sleep(2)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist Infos".format(len(searchedForLocalArtistInfo), db))
        localArtistInfo.save(data=searchedForLocalArtistInfo)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving [{0}] {1} Searched For Artist IDs".format(len(searchedForLocalArtistInfo), db))
localArtistInfo.save(data=searchedForLocalArtistInfo)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
searchedForLocalArtistInfo[artistPseudoID] = artistName    
localArtistInfo.save(data=searchedForLocalArtistInfo)

In [ ]:
localArtistInfo = [[ifile.stem for ifile in mio.dir.getRawModValDataDir(modVal).glob("*.p")] for modVal in range(100)]

In [ ]:
from listUtils import getFlatList
flatArtists = getFlatList(localArtistInfo)

In [ ]:
from pandas import Series
from copy import deepcopy
tmp = deepcopy(localArtistInfo)

In [ ]:

#Series(flatArtists)
knownArtistInfoData = possibleArtistsData[possibleArtistsData["ArtistPseudoID"].isin(flatArtists)]
localArtistInfo = knownArtistInfoData['name']
localArtistInfo.index = knownArtistInfoData["ArtistPseudoID"]

In [ ]:
x = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistInfo".format(db.lower()))
x.save(data=localArtistInfo.to_dict())

In [ ]:
artistNamesToGet = knownArtists[knownArtists.str.len() < 1].index
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))
artistNamesToGet = artistNamesToGet[~artistNamesToGet.isin(localArtistIDs.get().keys())]
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))

mmeDF = mdbio.getData()

unmatchedArtistNames = Series(mmeDF["ArtistName"].unique())
searchedForMasterArtists = masterArtists.get()
artistNamesToGet = unmatchedArtistNames[~unmatchedArtistNames.isin(searchedForMasterArtists.keys())]

print("{0} Search Results".format(db))
print("   Known Master Artist Names:    {0}".format(len(unmatchedArtistNames)))
print("   Known Spotify Artist Names:   {0}".format(len(searchedForMasterArtists)))
print("   Artist Names To Get:          {0}".format(len(artistNamesToGet)))
## localArtists.get()

In [ ]:
localArtists.get()['72663229469']

In [ ]:
files = mio.dir.getRawModValDataDir(0).glob("*.p")
7307726174100
16964600600

In [ ]:
list(files)[0]

In [ ]:
from lib import lastfm
aid = lastfm.MusicDBID()
aid.getArtistPseudoID('John Legend')
aid.getArtistID("https://www.last.fm/music/John+Legend")

In [ ]:
io.get("/Volumes/Piggy/Discog/artists-lastfm/0/albums/7307726174100.p")

In [ ]:
availableArtistData = lastfm.MusicDBIO(verbose=False).data.getSearchArtistData()


In [ ]:
knownArtistData = knownArtistData.drop_duplicates(subset=["url"], keep='first')


In [ ]:
lastfm.MusicDBIO(verbose=False).data.saveSearchArtistData(data=knownArtistData)

In [ ]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistIDsToGet = knownArtists[knownArtists.str.len() < 1].index
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))
artistIDsToGet = artistIDsToGet[~artistIDsToGet.isin(localArtistIDs.get().keys())]
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))

In [ ]:
artistIDsToGet

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "9:15pm")
n  = 0
searchedForArtistIDsData = localArtistIDsData.get()
searchedForArtistIDs = localArtistIDs.get()
searchedForErrors = errors.get()
for i,dbID in enumerate(artistIDsToGet):
    if searchedForArtistIDs.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    response = apiio.getArtistIDLookupResults(artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = "?"
        saveSearchedForErrors(searchedForErrors)
        apiio.sleep(5)
        continue
        
    searchedForArtistIDsData[dbID] = response
    searchedForArtistIDs[dbID] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
        saveSearchedForLocalArtistIDs(searchedForArtistIDs)
        saveSearchedForLocalArtistIDsData(searchedForArtistIDsData)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
saveSearchedForLocalArtistIDs(searchedForArtistIDs)
saveSearchedForLocalArtistIDsData(searchedForArtistIDsData)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

In [ ]:
from pandas import DataFrame, Series, concat

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistIDsDataFrame(aids):
    df = None
    if isinstance(aids,dict):
        df = DataFrame([v for k,v in aids.items()])[0].apply(Series) if len(aids) > 0 else None
    return df
        
def getResponseDataFrame(aids):
    df = getArtistIDsDataFrame(aids)
    if not isinstance(df,DataFrame):
        return None
    #df = DataFrame(response)
    df["followers"] = df["followers"].apply(getFollowers)
    df.index = df['sid']
    df.drop(['sid'], axis=1, inplace=True)
    return df

In [ ]:
aids = getSearchedForLocalArtistIDsData()
df   = getResponseDataFrame(aids)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getSearchArtistData()
    print("Found {0} Previous Artists".format(searchDF.shape[0]))
    searchDF = concat([searchDF,df])
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF = searchDF.sort_values(by='popularity', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

In [ ]:
saveSearchedForLocalArtistIDsData({})

# Move Local

In [ ]:
from mdblib.spotify import MusicDBIO
mio = MusicDBIO()

In [ ]:
from mdblib.spotify import moveLocalFiles

In [ ]:
moveLocalFiles()

# Download Album Data

## Find Albums To Get

In [ ]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistNamesToGet = knownArtists[~knownArtists.index.isin(searchedForAlbums.keys())]
artistNamesToGet = artistNamesToGet #.sample(frac=1)
print("   Artists IDs To Get:   {0}".format(len(artistNamesToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Albums".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "4:00pm")
n  = 0
searchedForAlbums = getSearchedForLocalAlbums()
searchedForErrors = getSearchedForErrors()
for i,(dbID,artistName) in enumerate(artistNamesToGet.iteritems()):    
    if searchedForAlbums.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    modVal=mio.mv.get(dbID)
    if mio.data.getRawArtistAlbumFilename(modVal, dbID).exists():
        searchedForAlbums[dbID] = artistName
        continue
    
    response = apiio.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = artistName
        saveSearchedForErrors(searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistAlbumData(data=response, modval=modVal, dbID=dbID)        
    searchedForAlbums[dbID] = artistName
    apiio.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
        saveSearchedForLocalAlbums(searchedForAlbums)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        apiio.sleep(30.0)
    
ts.stop()
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
saveSearchedForLocalAlbums(searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

In [ ]:
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
saveSearchedForLocalAlbums(searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

# Move Local Files

In [ ]:
from mdblib import spotify
spotify.moveLocalFiles()

# Parse What We Got

In [ ]:
%autoreload
from mdbutils import poolIO
mio = discogs.MusicDBIO(debug=False)
poolIO(parseFunction=mio.prd.parseAlbumData, modVals=range(100))

# Download Artist Info Data

## Determine Artists To Download

## Download Artist Info

# Download Artist Albums Data

## Determine Artists To Download

In [ ]:
spotifyArtists = io.get("/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
spotifyArtists = spotifyArtists.sort_values(by='popularity', ascending=False)

In [ ]:
searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = spotifyArtists[~spotifyArtists.index.isin(searchedForArtists.index)]['name']
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
from pandas import Series
meu = masterManualEntriesUtils()

knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    sids = [sid for sid in sids if len(sid) == 22]
    spotifyToGet.update({sid: name for sid in sids})
knownSpotify = Series(spotifyToGet)
print("Found {0} Known Spotify Artists".format(len(knownSpotify)))

searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = knownSpotify[~knownSpotify.index.isin(searchedForArtists.index)]
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
from timeUtils import timestat, termTime

dbIO = dbg.getDBIO("Spotify")
api  = apig.getDBAPIData("Spotify")

searchedForArtists = io.get("spotifySearchedForArtists.p")
errs = io.get("spotifyErrorsSearchedForArtists.p")
ts = timestat("Downloading Spotify Data")
#tt = termTime("today", "10:00pm")
tt = termTime("tomorrow", "10:00pm")
#tt = termTime("tomorrow", "7:00am")
#tt = termTime("today", "9:30am")
n  = 0
for i,(dbID,artistName) in enumerate(artistNames.iteritems()):    
    if searchedForArtists.get(dbID) is not None:
        continue
    if errs.get(dbID) is not None:
        continue
    if not dbIO.isArtistAlbumsKnown(dbID) or True:
        response = api.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        errs[dbID] = artistName
        io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")
        api.sleep(5)
        continue
    dbIO.saveArtistAlbums(dbID, response)
    searchedForArtists[dbID] = artistName
    api.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        print("="*150)
        api.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        api.sleep(30.0)
    
ts.stop()
print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Spotify Errors For Artists".format(len(errs)))
    io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")

# Extra

In [ ]:
  
    api.sleep(7.5)
    n += 1
    
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        api.sleep(7.5)
    
    if (i+1) % 25 == 0:
        sleep(30.0)

In [ ]:
######################################################
# Juypter
######################################################
%load_ext autoreload
%autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("""<style>div.output_area{max-height:10000px;overflow:scroll;}</style>"""))

###########################################################################
## Warnings
###########################################################################
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 


###########################################################################
## Utils
###########################################################################
from ioUtils import getFile, saveFile
from webUtils import getHTML
from searchUtils import findExt
from fsUtils import setFile, isFile, fileUtil
from fileUtils import getBaseFilename
from timeUtils import timestat
from fileIO import fileIO
from dbUtils import utilsSpotifyCharts


###########################################################################
## General
###########################################################################
import urllib
from glob import glob
from time import sleep
from io import StringIO
from pandas import Series,DataFrame,concat,read_csv,date_range
from datetime import datetime
from urllib.parse import unquote, urlparse
from pathlib import PurePosixPath



######################################################
# Versions
######################################################
import sys
print("Python: {0}".format(sys.version))
import datetime as dt
start = dt.datetime.now()

print("Notebook Last Run Initiated: "+str(start))

# Global

In [ ]:
io = fileIO()
from masterDBGate import masterDBGate
mdbGate = masterDBGate()

from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

from spotipy.oauth2 import SpotifyClientCredentials
import spotipy
auth_manager=SpotifyClientCredentials(client_id="61e441c3b90c4873aa0e6b9582564f95", client_secret="ae0d0f968bf443fdac1d9ac6ef65fc0f")
sp = spotipy.Spotify(auth_manager=auth_manager)

# Spotify API

In [ ]:
# !pip install spotipy

In [ ]:
from artistSpotify import artistSpotify
#from artistIDBase import artistIDSpotify
from dbBase import dbBase

from fsUtils import dirUtil,fileUtil
from artistIDBase import dbArtistID
from artistModValue import artistModValue
from masterArtistNameCorrection import masterArtistNameCorrection


##################################################################################################################
# Base Class
##################################################################################################################
class dbSpotify:
    def __init__(self):
        self.db     = "Spotify"
        self.disc   = dbBase(self.db.lower())
        self.artist = artistSpotify(self.disc)
        self.aid    = dbArtistID(self.db)

        self.getpsid   = self.aid.getpsid
        self.getModVal = self.aid.getModVal
        
        self.io = fileIO()
        self.manc = masterArtistNameCorrection()
        
        
    def getSearchDir(self, artistName):
        psID = self.getpsid(self.manc.clean(artistName))
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(psID)))
        searchDir = dirUtil(modValDir).join("search")
        return searchDir
        
    def getArtistSearchSavename(self, artistName):
        psID = self.getpsid(self.manc.clean(artistName))
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(psID)))
        searchDir = dirUtil(modValDir).join("search")
        return fileUtil(searchDir).join("{0}.p".format(psID))
    
    def saveArtistSearch(self, artistName, artistSearch):
        self.io.save(idata=artistSearch, ifile=self.getArtistSearchSavename(artistName))
        
        
    def getArtistAlbumsSavename(self, artistID):
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(artistID)))
        return fileUtil(modValDir).join("{0}.p".format(artistID))
    
    def saveArtistAlbums(self, artistID, artistAlbums):
        self.io.save(idata=artistAlbums, ifile=self.getArtistAlbumsSavename(artistID))

In [ ]:
import requests
from time import sleep

class apiIO:
    def __init__(self, name):
        self.name = name
        self.code = None
        self.url = None
        self.response = None
        
    def get(self, url):
        self.url       = url
        try:
            self.response  = requests.get(url)
        except:
            self.response  = None
            self.code      = 0
            return {}
            
        self.code      = self.response.status_code
        
        try:
            json_data      = self.response.json() if self.response.status_code == 200 else {}
        except:
            json_data      = {}
        return json_data
    
    def getResponse(self):
        return self.response
    
    def getURL(self):
        return self.url
    
    def getStatus(self):
        return self.code
    
    def sleep(self, value):
        sleep(value)

In [ ]:
def getArtistTrackResults(artistName, artistID, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    searchResults  = []
    offset         = 0
    sleep(0.5)
    requestResult  = sp.artist_top_tracks(artistID, limit=limit, offset=offset)
    offset += limit
    
    if requestResult is None:
        return None
    totalResults   = requestResult.get('total', -1)
    nextURL        = requestResult.get('next', None)
    try:
        searchResults += requestResult['items']
    except:
        return None
    print("   ===> {0: <8}   {1}".format("[{0}]".format(totalResults), len(searchResults)), end=" ")
    while nextURL is not None:
        requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
        offset += limit
        try:
            searchResults += requestResult['items']
        except:
            return None
        nextURL        = requestResult.get('next', None)
        if nextURL:
            #print("{0}".format(len(searchResults)), end="")
            print(".", end="")
            sleep(1.0)
    print(" {0}".format(len(searchResults)))
    return searchResults

In [ ]:
def getArtistAlbumResults(artistName, artistID, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    searchResults  = []
    offset         = 0
    sleep(0.5)
    requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
    offset += limit
    
    if requestResult is None:
        return None
    totalResults   = requestResult.get('total', -1)
    href           = requestResult.get('href')
    artistID       = PurePosixPath(unquote(urlparse(href).path)).parts[3]
    nextURL        = requestResult.get('next', None)
    try:
        searchResults += requestResult['items']
    except:
        return None
    print("   ===> {0: <8}   {1}".format("[{0}]".format(totalResults), len(searchResults)), end=" ")
    while nextURL is not None:
        sleep(3.5)
        requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
        offset += limit
        try:
            searchResults += requestResult['items']
        except:
            return None
        nextURL        = requestResult.get('next', None)
        if nextURL:
            #print("{0}".format(len(searchResults)), end="")
            print(".", end="")
    print(" {0}".format(len(searchResults)))
    
    albums = [spotifyAlbumRecord(album).get() for album in searchResults]
    retval = {'href': href, 'artistID': artistID, 'albums': albums}
    return retval

In [ ]:
def getArtistSearchResults(artistName, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    sleep(0.25)
    result = sp.search(artistName, limit=limit, type='artist')
    
    artists = result.get('artists', {}) if isinstance(result,dict) else {}
    href    = artists.get('href')
    total   = artists.get('total')
    nextURL = artists.get('next')
    prevURL = artists.get('previous')
    items   = artists.get('items', [])
    retval  = [spotifyArtistRecord(item).get() for item in items]
    print(len(retval))
    
    return retval

# Artist Search

In [ ]:
from glob import glob
from masterUtils import masterUtils
mu = masterUtils()
disc = mu.getDisc("Spotify")
ts = timestat("Finding Search Files")
files = glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")
ts.stop()

In [ ]:
from fileIO import fileIO
io = fileIO()
#io.save(idata=files, ifile="spotifyFiles.p")
files = io.get("spotifyFiles.p")
len(files)

In [ ]:
from fsUtils import fileUtil, mkDir, dirUtil, moveFile
dbIO = dbSpotify()
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    artistName = fileUtil(ifile).basename
    searchDir = dbIO.getSearchDir(artistName)
    if not searchDir.exists():
        print("Making {0}".format(searchDir))
        mkDir(searchDir)
    savename = dbIO.getArtistSearchSavename(artistName=artistName)
    moveFile(ifile,savename)
    
    if (i+1) % 10000 == 0 or (i+1) == 1000 or (i+1) == 100:
        ts.update(n=i+1, N=len(files))
ts.stop()

In [ ]:
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
ts = timestat("Getting Searched For Artists")
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
ts.stop()
print("Found {0} Searched For Artists".format(len(searchedForArtists)))

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
ts = timestat("Getting Artists To Download")
mmeDF = meu.getDF()
artistNames = mmeDF["ArtistName"].apply(manc.clean)
print("Found {0} Artists".format(artistNames.shape[0]))
artistNamesToDownload = artistNames[~artistNames.isin(searchedForArtists)]
print("Found {0} Artists To Download".format(artistNamesToDownload.shape[0]))
ts.stop()

In [ ]:
toget = {}
for i,(idx,artistName) in enumerate(artistNamesToDownload.iteritems()):
    if i >= 2500:
        break
    savefile = fileUtil("spotify/artists").join("{0}.p".format(manc.clean(artistName)))
    if savefile.exists():
        continue
    toget[savefile] = artistName
print("Need To Download {0} Artists".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,artistName) in enumerate(toget.items()):
    if savefile.exists():
        continue
    if nErr >= 5:
        break
    result = getArtistSearchResults(artistName)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60*nErr)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(2.5)
    
    if (i+1) % 25 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
    
    if (i+1) % 100 == 0:
        sleep(120)
ts.stop()

## Artist Results

In [ ]:
from artistModValue import artistModValue
amv = artistModValue()

# Move Artist Files

In [ ]:
from fsUtils import dirUtil,fileUtil,moveFile

files = glob("spotify/artists/*.p")
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    if (i+1) % 1000 == 0:
        ts.update(n=i+1,N=len(files))
        
    src = fileUtil(ifile)
    dst = dirUtil("/Volumes/Piggy/Discog/artists-spotify/artists").join(src.name)
    if dst.exists():
        continue
    moveFile(src.path, dst)

In [ ]:
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
print("Found {0} Searched For Artists".format(len(searchedForArtists)))

# Move Album Files

In [ ]:
from fsUtils import dirUtil,fileUtil,moveFile,mkDir
from artistModValue import artistModValue
amv = artistModValue()

files = glob("spotify/albums/*.p")
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    if (i+1) % 1000 == 0:
        ts.update(n=i+1,N=len(files))
        
    src = fileUtil(ifile)
    albumsData = io.get(ifile)
    artistID = albumsData['artistID']
    modValue = amv.getModVal(artistID)
    modDir = dirUtil("/Volumes/Piggy/Discog/artists-spotify/").join(str(modValue))
    if not modDir.exists():
        mkDir(modDir)
    dst = dirUtil(modDir).join("{0}.p".format(artistID))
    if dst.exists():
        continue
    print("  ==>  {0}".format(dst))
    moveFile(src.path, dst)

# Create Master Artist ID List

In [ ]:
from glob import glob
files = glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")
artistsData = []
N = len(files)
ts = timestat("Loading {0} Spotify Artist Search Files".format(N))
for i,ifile in enumerate(files):    
    artistsData += io.get(ifile)
    if (i+1) % 5000 == 0 or (i+1) == 1000:
        ts.update(n=i+1, N=N)
ts.stop()

In [ ]:
def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

df = DataFrame(artistsData)
df["followers"] = df["followers"].apply(getFollowers)
df.index = df['sid']
df.drop(['sid'], axis=1, inplace=True)
df = df[~df.index.duplicated(keep='first')]
df = df.sort_values(by='popularity', ascending=False)

print("Saving {0} Spotify Artists Data".format(df.shape[0]))
io.save(idata=df, ifile="/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
df.head(10)

In [ ]:
print(df.shape)

# Artist Albums

In [ ]:
spotifyArtists = io.get("/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
spotifyArtists = spotifyArtists.sort_values(by='popularity', ascending=False)

## Download Albums Data

In [ ]:
spotifyArtists['popularity'].hist(bins=100, log='y')

In [ ]:
spotifyArtists[spotifyArtists["popularity"] >= 60].shape

In [ ]:
knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    for sid in sids:
        if len(sid) == 22:
            spotifyToGet[sid] = name
spotifyToGet = Series(spotifyToGet)
#spotifyToGet = knownSpotify["ArtistName"]
#spotifyToGet.index = list(knownSpotify["Spotify"].values)
#spotifyToGet.name = "name"
#spotifyToGet

In [ ]:
toget = {}
from artistModValue import artistModValue
from masterUtils import masterUtils
from fsUtils import dirUtil, mkDir
mu   = masterUtils()
amv  = artistModValue()
disc = mu.discs["Spotify"]


#spotifyToGet = spotifyArtists[spotifyArtists["popularity"] >= 60]
#print("Found {0} Spotify Artists".format(spotifyToGet.shape[0]))
#for i,(artistID,artistName) in enumerate(spotifyToGet["name"].iteritems()):

print("Found {0} Spotify Artists".format(spotifyToGet.shape[0]))
nKnown = 0
for i,(artistID,artistName) in enumerate(spotifyToGet.iteritems()):
    modVal   = amv.getModVal(artistID)
    modDir   = dirUtil(disc.getArtistsDir()).join(str(modVal))
    if not modDir.exists():
        print("Making {0}".format(modDir))
        mkDir(modDir)
    savefile = dirUtil(modDir).join("{0}.p".format(artistID))
    if savefile.exists():
        nKnown += 1
        continue
    #print("{0: <25}{1: <20} ({2})".format(artistName,artistID,modVal))
    toget[savefile] = (artistName,artistID)
    if len(toget) >= 2500:
        break
print("Found {0} Known Artists".format(nKnown))
print("Need To Download {0} Artist Albums".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,(artistName,artistID)) in enumerate(toget.items()):
    if nErr >= 2:
        break
    result = getArtistAlbumResults(artistName=artistName, artistID=artistID)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(7.5)
    
    if (i+1) % 5 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(15.0)
        print("")
    
    if (i+1) % 25 == 0:
        sleep(30.0)
ts.stop()

## Determine Artists To Download

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    for sid in sids:
        if len(sid) == 22:
            spotifyToGet[sid] = name
knownSpotify = Series(spotifyToGet)
print("Found {0} Known Spotify Artists".format(len(knownSpotify)))

searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = knownSpotify[~knownSpotify.index.isin(searchedForArtists.index)]
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
io.save(idata={}, ifile="spotifyErrorsSearchedForArtists.p")

In [ ]:
stop = 150
dbIO = dbSpotify()
api  = spotifyAPIIO()
ts   = timestat("Getting Artist Data From Spotify API")
n    = 0

searchedForArtists = io.get("spotifySearchedForArtists.p")
errs = io.get("spotifyErrorsSearchedForArtists.p")
for i,(artistID,artistName) in enumerate(artistNames.iteritems()):
    if searchedForArtists.get(artistID) is not None:
        continue
    if errs.get(artistID) is not None:
        continue
    savename = dbIO.getArtistAlbumsSavename(artistID)
    if savename.exists():
        searchedForArtists[artistID] = artistName 
        continue
        
    #print(artistID,artistName,savename)
    response = api.getArtistAlbums(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        errs[artistID] = artistName
        io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")
        sleep(5)
        continue
    dbIO.saveArtistAlbums(artistID=artistID, artistAlbums=response)
    searchedForArtists[artistID] = artistName    
    api.sleep(7.5)
    n += 1
    
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        api.sleep(7.5)
    
    if (i+1) % 25 == 0:
        sleep(30.0)
    
ts.stop()
print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Spotify Error Artists".format(len(errs)))
    io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")

In [ ]:
if False:
    searchedForArtists = io.get("spotifySearchedForArtists.p")
    print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

    ts = timestat("Finding Spotify Saves")
    first = True
    for i,(artistID,artistName) in enumerate(spotifyToGet.iteritems()):
        if searchedForArtists.get(artistID) is not None:
            continue
        if first:
            print("Start: {0}".format(i))
            first = False
        if dbIO.getArtistAlbumsSavename(artistID).exists():
            searchedForArtists[artistID] = artistName
            if len(searchedForArtists) % 5000 == 0:
                break
        else:
            break

        if (i+1) % 100 == 0:
            ts.update(n=i+1,N=len(spotifyToGet))        
            #io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")

    ts.stop()
    print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
    io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")

In [ ]:
spotifyArtists.head()

In [ ]:
from dbArtistsBase import dbArtistsBase
from fileUtils import getBaseFilename
from fsUtils import isFile, setFile, setDir
from ioUtils import getFile, saveFile
from timeUtils import timestat
from sys import prefix
import urllib
from time import sleep
from webUtils import getHTML
from pandas import Series
    
from fileIO import fileIO
from fsUtils import fileUtil

In [ ]:
from dbArtistsParse import dbArtistsSpotifyAPI
from dbArtistsSpotify import dbArtistsSpotify
dbAP = dbArtistsSpotifyAPI(dbArtistsSpotify())

In [ ]:
for modVal in range(100):
    dbAP.parse(modVal=modVal, expr='< 0 Days', force=False)
    
#for modVal in range(100):
#    dbAP.parseSearch(modVal, expr='< 0 Days', force=False)

In [ ]:
from artistModValue import artistModValue
amv = artistModValue()
modVal = 91
idx = artistSearchData.reset_index()['sid'].apply(amv.getModVal) == modVal
idx.index = artistSearchData.index
artistSearchData[idx]

In [ ]:
artistSearchData.head()

In [ ]:
art = artistSpotify()
art.getData(inputData).show()

In [ ]:
io.get("/Users/tgadfort/Music/Discog/artists-spotify-db/91-DB.p").get('1uNFoZAHBGtllmzznpCI3s').show()

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
mmeDF = meu.getDF()

In [ ]:

toget = {}
for i,(artistID,artistName) in enumerate(spotifyArtists["name"].iteritems()):
    if i >= 5:
        break
    savefile = fileUtil("spotify/albums").join("{0}--{1}.p".format(artistID,manc.clean(artistName)))
    if savefile.exists():
        pass
        #continue
    toget[savefile] = (artistName,artistID)
print("Need To Download {0} Artist Albums".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,(artistName,artistID)) in enumerate(toget.items()):
    if nErr >= 2:
        break
    result = getArtistAlbumResults(artistName=artistName, artistID=artistID)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(3.0)
    
    if (i+1) % 5 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
ts.stop()

## Artist Albums Results

In [ ]:
from glob import glob
files = glob("spotify/albums/*.p")
artistAlbumsData = {}
for ifile in files:
    albumsData   = io.get(ifile)
    artistID     = albumsData['artistID']
    artistAlbums = albumsData['albums']
    if artistAlbumsData.get(artistID) is not None:
        raise ValueError("Already have data for {0}".format(artistID))
    artistAlbumsData[artistID] = artistAlbums

In [ ]:
retval = getArtistAlbumResults(artistID='46sPgFJpEcoxUJNr1ouR9G', artistName='dummy')

In [ ]:
s = Series(artistAlbumsData)

In [ ]:
s.apply(len)